In [2]:
# ============================================================
# CELL 1 — Install
# ============================================================
!pip install plotly pandas --quiet

# ============================================================
# CELL 2 — Imports
# ============================================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print('✅ Libraries loaded')

# ============================================================
# CELL 3 — Upload your CSV to Colab, then load it
# Drag and drop your file into the Colab file panel (left sidebar)
# OR run: from google.colab import files; files.upload()
# Then update the filename below if needed
# ============================================================
df = pd.read_csv('/content/Public Transport accessbility  - Sheet1.csv')
df.columns = ['Entity', 'Code', 'Year', 'Access']
df['Year'] = df['Year'].astype(int)
df['Access'] = pd.to_numeric(df['Access'], errors='coerce')
df.dropna(subset=['Access'], inplace=True)

print(f'✅ Loaded {len(df):,} rows')
print(f'   Years: {sorted(df["Year"].unique())}')
print(f'   Countries: {df["Entity"].nunique()}')
df.head()

# ============================================================
# CELL 4 — GRAPH 1: World Choropleth Map
# ============================================================
latest = df.sort_values('Year').groupby('Entity').last().reset_index()

fig1 = px.choropleth(
    latest,
    locations='Code',
    color='Access',
    hover_name='Entity',
    hover_data={'Year': True, 'Access': ':.1f', 'Code': False},
    color_continuous_scale=[
        [0.0,  '#7f0000'],
        [0.25, '#d73027'],
        [0.5,  '#f7f7f7'],
        [0.75, '#4575b4'],
        [1.0,  '#053061'],
    ],
    range_color=[0, 100],
    labels={'Access': '% with Access'},
    title='🌍 Population with Convenient Access to Public Transport (Most Recent Year)',
    height=550,
)
fig1.update_layout(
    coloraxis_colorbar=dict(title='% Access', ticksuffix='%', thickness=15, len=0.6),
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    margin=dict(t=60, b=10, l=10, r=10),
    font=dict(family='Arial', size=13),
    title_x=0.5,
)
fig1.add_annotation(
    text="Source: Our World in Data — UN SDG Indicator 11.2.1",
    xref="paper", yref="paper", x=0.5, y=-0.05,
    showarrow=False, font=dict(size=11, color='gray'), xanchor='center'
)
fig1.show()
fig1.write_html('/content/choropleth_map.html')
print('✅ Choropleth map saved')

# ============================================================
# CELL 5 — GRAPH 2: Line Chart — Pre/Post COVID Trends
# ============================================================
multi = df.groupby('Entity').filter(lambda x: len(x) >= 2)
highlight = ['United States', 'United Kingdom', 'Brazil', 'China', 'India', 'Germany', 'Nigeria']
global_avg = df.groupby('Year')['Access'].mean().reset_index()

fig2 = go.Figure()

# Faint background lines for all countries
for country, group in multi.groupby('Entity'):
    if country not in highlight:
        fig2.add_trace(go.Scatter(
            x=group['Year'], y=group['Access'],
            mode='lines', name=country,
            line=dict(color='lightgray', width=1),
            showlegend=False,
            hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>Access: %{{y:.1f}}%<extra></extra>'
        ))

# Highlighted countries
colors = ['#E85D26', '#3B82F6', '#8B5CF6', '#10B981', '#F59E0B', '#EF4444', '#06B6D4']
for i, country in enumerate(highlight):
    group = df[df['Entity'] == country]
    if len(group) > 0:
        fig2.add_trace(go.Scatter(
            x=group['Year'], y=group['Access'],
            mode='lines+markers', name=country,
            line=dict(color=colors[i % len(colors)], width=3),
            marker=dict(size=8),
            hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>Access: %{{y:.1f}}%<extra></extra>'
        ))

# Global average line
fig2.add_trace(go.Scatter(
    x=global_avg['Year'], y=global_avg['Access'],
    mode='lines+markers', name='🌍 Global Average',
    line=dict(color='black', width=3, dash='dash'),
    marker=dict(size=8, symbol='diamond'),
    hovertemplate='<b>Global Average</b><br>Year: %{x}<br>Access: %{y:.1f}%<extra></extra>'
))

# COVID marker
fig2.add_vline(x=2020, line_dash='dot', line_color='red', line_width=2)
fig2.add_annotation(x=2020, y=95, text='COVID-19', showarrow=False,
                    font=dict(color='red', size=12), xshift=30)

fig2.update_layout(
    title=dict(text='📉 Public Transport Accessibility 2016–2022 (Pre/Post COVID)', x=0.5),
    xaxis=dict(title='Year', tickmode='array', tickvals=sorted(df['Year'].unique())),
    yaxis=dict(title='% of Population with Convenient Access', range=[0, 105], ticksuffix='%'),
    legend=dict(x=1.01, y=1, bordercolor='lightgray', borderwidth=1),
    height=550,
    font=dict(family='Arial', size=13),
    hovermode='x unified',
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig2.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.show()
fig2.write_html('/content/line_chart.html')
print('✅ Line chart saved')

# ============================================================
# CELL 6 — GRAPH 3: Bar Chart — Year Comparison
# ============================================================
years = sorted(df['Year'].unique())
year_a = years[0]   # earliest year
year_b = years[-1]  # latest year

has_a = set(df[df['Year'] == year_a]['Entity'])
has_b = set(df[df['Year'] == year_b]['Entity'])
both = list(has_a & has_b)

if len(both) > 0:
    df_a = df[(df['Year'] == year_a) & (df['Entity'].isin(both))][['Entity','Access']].rename(columns={'Access': str(year_a)})
    df_b = df[(df['Year'] == year_b) & (df['Entity'].isin(both))][['Entity','Access']].rename(columns={'Access': str(year_b)})
    compare = df_a.merge(df_b, on='Entity')
    compare['Change'] = compare[str(year_b)] - compare[str(year_a)]
    compare = compare.sort_values('Change')

    fig3 = go.Figure()
    fig3.add_trace(go.Bar(
        name=str(year_a), x=compare['Entity'], y=compare[str(year_a)],
        marker_color='#93C5FD',
        hovertemplate='<b>%{x}</b><br>'+str(year_a)+': %{y:.1f}%<extra></extra>'
    ))
    fig3.add_trace(go.Bar(
        name=str(year_b), x=compare['Entity'], y=compare[str(year_b)],
        marker_color='#1D4ED8',
        hovertemplate='<b>%{x}</b><br>'+str(year_b)+': %{y:.1f}%<extra></extra>'
    ))
    fig3.update_layout(
        title=dict(text=f'📊 Public Transport Accessibility: {year_a} vs {year_b} by Country', x=0.5),
        barmode='group',
        xaxis=dict(title='Country', tickangle=-40),
        yaxis=dict(title='% with Convenient Access', ticksuffix='%', range=[0, 115]),
        legend=dict(x=0.01, y=0.99),
        height=550,
        font=dict(family='Arial', size=12),
        plot_bgcolor='white',
        paper_bgcolor='white',
    )
    fig3.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
    fig3.show()
    fig3.write_html('/content/bar_chart_comparison.html')
    print(f'✅ Bar chart saved — {len(both)} countries with both {year_a} & {year_b} data')
else:
    print(f'⚠️  No countries have both {year_a} and {year_b} — check your data years')

# ============================================================
# CELL 7 — Download All
# ============================================================
from google.colab import files
for path in ['/content/choropleth_map.html',
             '/content/line_chart.html',
             '/content/bar_chart_comparison.html']:
    try:
        files.download(path)
    except Exception as e:
        print(f'⚠️  Could not download {path}: {e}')
print('✅ Done!')

✅ Libraries loaded
✅ Loaded 141 rows
   Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2021), np.int64(2022)]
   Countries: 137


✅ Choropleth map saved


✅ Line chart saved
⚠️  No countries have both 2016 and 2022 — check your data years


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⚠️  Could not download /content/bar_chart_comparison.html: Cannot find file: /content/bar_chart_comparison.html
✅ Done!
